# Curtaining score - batch (FOV-corrected wedge + curtain-fraction map)

Runs the **FOV-corrected wedge method** over **every `.tif` in a folder** (searched recursively)
and writes **one CSV** with all the scores. Optional per-image, per-step raw TIFFs.

Pipeline per image:

1. **FFT** of the (masked) image.
2. **FOV-corrected angular wedge** around the horizontal frequency axis - the wedge angle is
   measured in physical (cycles/px) coordinates, so the score does not depend on the image
   width/height (field of view).
3. **Inverse FFT** of the wedge -> a stripe-only reconstruction (the curtain component).
4. **Scores**: FOV-corrected wedge energy ratio, plus a spatial **curtain map**.

## Step 0 - Configuration

Update `INPUT_FOLDER` and `OUTPUT_DIR` before running the notebook.

- **`INPUT_FOLDER`** - folder containing the TIFF images; subfolders are searched recursively.
- **`OUTPUT_DIR`** - destination for the CSV and optional diagnostic TIFFs.
- **Wedge parameters** - control the angular frequency band used to detect curtaining.
- **Curtain-map parameters** - control local texture estimation and the curtain threshold.
- **`SAVE_PICTURES`** - save each processing step under `curtaining_diagnostics/<image>/`.


In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage

# ----------------------------------------------------------------- inputs ---
# Update these paths before running the notebook.
INPUT_FOLDER = "path/to/input"
OUTPUT_DIR = "path/to/output"

# ------------------------------------------------------- wedge parameters ---
WEDGE_ANGLE_DEG = 5.0
R0_FRAC         = 0.02
R1_FRAC         = 1.0
FOV_CORRECT     = True     # measure wedge angle in physical (cycles/px) coords -> size-robust
USE_GRADIENT    = False
TILE            = 256      # tile size (px) for the local heatmap panel

# --------------------------------------------------------- valid-pixel mask ---
USE_NONZERO_MASK = True
MASK_THRESHOLD   = 0.0
MIN_VALID_FRAC   = 0.5

# ------------------------------------------------------ curtain-map params ---
ENV_WIN             = 15    # px, window for the local RMS envelopes
DETREND_WIN         = 63    # px, high-pass window that defines "total local texture"
CURTAIN_FRAC_THRESH = 0.35  # curtain pixel if curtain RMS > this fraction of texture RMS
BORDER_TRIM         = 8     # erode valid mask by this many px before scoring (drops edge artefacts)

# --------------------------------------------------------------- output ---
SAVE_PICTURES = True   # save raw per-step TIFFs (one subfolder per image)
CSV_NAME      = "curtaining_batch_scores.csv"

# ------------------------------------------------------------------ setup ---
OUT_DIR = OUTPUT_DIR if OUTPUT_DIR else INPUT_FOLDER
os.makedirs(OUT_DIR, exist_ok=True)
PIC_DIR = os.path.join(OUT_DIR, "curtaining_diagnostics")
if SAVE_PICTURES:
    os.makedirs(PIC_DIR, exist_ok=True)

print(f"Input folder : {INPUT_FOLDER}")
print(f"Output folder: {OUT_DIR}")
print(f"Wedge        : +/-{WEDGE_ANGLE_DEG} deg, band {R0_FRAC}-{R1_FRAC}, FOV_CORRECT={FOV_CORRECT}")
print(f"Curtain map  : env_win={ENV_WIN}, detrend_win={DETREND_WIN}, "
      f"frac_thresh={CURTAIN_FRAC_THRESH}, border_trim={BORDER_TRIM}")
print(f"Save TIFFs   : {SAVE_PICTURES}")


## Step 1 - Analysis functions

`analyze_image` returns every score for one image; `find_tifs` walks the folder skipping
segmentation/mask companion files.


In [ ]:
def find_tifs(root, skip_tokens=("_Probabilities", "_Simple Segmentation", "_full_cell")):
    """Walk `root` and return sorted .tif files, skipping companion/mask files."""
    files = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith((".tif", ".tiff")):
                continue
            if any(tok in fname for tok in skip_tokens):
                continue
            files.append(os.path.join(dirpath, fname))
    return sorted(files)


def load_image(path):
    """Load a TIFF as a single 2-D float64 image (channel 0 / middle slice for 3-D)."""
    img = np.asarray(tifffile.imread(path))
    if img.ndim == 3:
        img = img[..., 0] if img.shape[-1] <= 4 else img[img.shape[0] // 2]
    return img.astype(np.float64)


def valid_mask(image, use_mask=USE_NONZERO_MASK, threshold=MASK_THRESHOLD):
    img = np.asarray(image)
    if not use_mask:
        return np.ones(img.shape, dtype=bool)
    return img > threshold


def prep_for_fft(image, valid):
    """Fill invalid pixels with the valid-region mean so they add no spurious frequencies."""
    img = np.asarray(image, dtype=np.float64)
    if valid is None or valid.all() or not valid.any():
        return img
    out = img.copy()
    out[~valid] = img[valid].mean()
    return out


def compute_fft(image, use_gradient=False):
    img = np.asarray(image, dtype=np.float64)
    if use_gradient:
        img = ndimage.sobel(img, axis=1, mode="nearest")
    img = img - img.mean()
    F = np.fft.fftshift(np.fft.fft2(img))
    return F, np.abs(F) ** 2


def _radius_angle(shape, fov_correct=FOV_CORRECT):
    """Normalised radius (0-1) and angle-from-horizontal (deg). FOV correction scales the
    frequency axes by image size so the wedge is measured in cycles/px, not pixels."""
    ny, nx = shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.ogrid[:ny, :nx]
    dy, dx = (yy - cy).astype(float), (xx - cx).astype(float)
    if fov_correct:
        dy = dy / ny
        dx = dx / nx
    radius = np.sqrt(dx ** 2 + dy ** 2)
    rmax = radius.max()
    radius = radius / rmax if rmax > 0 else radius
    angle = np.degrees(np.arctan2(np.abs(dy), np.abs(dx)))
    return radius, angle


def wedge_mask(shape, theta_deg=WEDGE_ANGLE_DEG, r0_frac=R0_FRAC, r1_frac=R1_FRAC,
               fov_correct=FOV_CORRECT):
    radius, angle = _radius_angle(shape, fov_correct)
    in_band = (radius >= r0_frac) & (radius <= r1_frac)
    return in_band & (angle <= theta_deg)


def annulus_mask(shape, r0_frac=R0_FRAC, r1_frac=R1_FRAC, fov_correct=FOV_CORRECT):
    radius, _ = _radius_angle(shape, fov_correct)
    return (radius >= r0_frac) & (radius <= r1_frac)


def recover_stripes(F, mask):
    """Inverse-FFT of just the wedge -> the curtain (stripe-only) component of the image."""
    Fw = np.zeros_like(F)
    Fw[mask] = F[mask]
    return np.real(np.fft.ifft2(np.fft.ifftshift(Fw)))


def curtain_fraction_map(stripes, imgf, env_win=ENV_WIN, detrend_win=DETREND_WIN):
    """Local fraction of texture explained by curtaining, in [0, 1].

    Numerator   : local RMS of the wedge/curtain component.
    Denominator : local RMS of the total high-frequency texture (image minus a smooth trend).
    Both scale with local contrast, so the ratio is dimensionless and scale-invariant."""
    env_curtain = np.sqrt(np.clip(ndimage.uniform_filter(stripes ** 2, env_win, mode="nearest"), 0, None))
    detrended = imgf - ndimage.uniform_filter(imgf, detrend_win, mode="nearest")
    env_total = np.sqrt(np.clip(ndimage.uniform_filter(detrended ** 2, env_win, mode="nearest"), 0, None))
    return env_curtain / (env_total + 1e-9)


def score_valid(valid, border_trim=BORDER_TRIM):
    """Valid mask eroded by `border_trim` px so edge/ringing artefacts are not scored."""
    if border_trim and border_trim > 0:
        return ndimage.binary_erosion(valid, iterations=int(border_trim))
    return valid


def curtain_mask_and_pct(frac, valid, thresh=CURTAIN_FRAC_THRESH):
    v = valid
    if v.sum() == 0:
        return np.zeros_like(frac, dtype=bool), 0.0
    mask = (frac > thresh) & v
    return mask, 100.0 * float(mask.sum()) / float(v.sum())


def analyze_patch_ratio(patch):
    v = valid_mask(patch)
    pf = prep_for_fft(patch, v)
    _, pw = compute_fft(pf, use_gradient=USE_GRADIENT)
    ae = float(pw[annulus_mask(patch.shape)].sum())
    return 100.0 * float(pw[wedge_mask(patch.shape)].sum()) / ae if ae > 0 else np.nan


def local_heatmap(image, tile=TILE, min_valid_frac=MIN_VALID_FRAC):
    img = np.asarray(image, dtype=np.float64)
    ny, nx = img.shape
    rs = list(range(0, ny, tile)); cs = list(range(0, nx, tile))
    hmap = np.full((len(rs), len(cs)), np.nan)
    for i, y0 in enumerate(rs):
        for j, x0 in enumerate(cs):
            patch = img[y0:y0 + tile, x0:x0 + tile]
            if patch.shape[0] < 8 or patch.shape[1] < 8:
                continue
            if USE_NONZERO_MASK and (patch > MASK_THRESHOLD).mean() < min_valid_frac:
                continue
            hmap[i, j] = analyze_patch_ratio(patch)
    return hmap


def analyze_image(image, frac_thresh=CURTAIN_FRAC_THRESH):
    """Full FOV-corrected wedge + curtain-fraction analysis of one image."""
    img = np.asarray(image, dtype=np.float64)
    ny, nx = img.shape
    valid = valid_mask(img)
    imgf = prep_for_fft(img, valid)
    F, power = compute_fft(imgf, use_gradient=USE_GRADIENT)
    wmask = wedge_mask(img.shape); amask = annulus_mask(img.shape)

    stripes = recover_stripes(F, wmask)

    annE = float(power[amask].sum()); wE = float(power[wmask].sum())
    ratio = 100.0 * wE / annE if annE > 0 else np.nan
    std_score = float(np.std(stripes[valid])) if valid.any() else float(np.std(stripes))

    frac = curtain_fraction_map(stripes, imgf)
    vscore = score_valid(valid)
    curtain_mask, pct = curtain_mask_and_pct(frac, vscore, frac_thresh)
    mean_frac = float(frac[vscore].mean()) if vscore.any() else np.nan

    return {
        "height": int(ny), "width": int(nx), "aspect_ratio": float(nx / ny),
        "curtaining_std": std_score,
        "wedge_energy_ratio": float(ratio) if np.isfinite(ratio) else np.nan,
        "curtain_area_pct": float(pct),
        "mean_curtain_fraction": mean_frac,
        "valid_frac": float(valid.mean()),
        # arrays for optional pictures
        "_log_power": np.log1p(power), "_wedge_log_power": np.log1p(power * wmask),
        "_stripes": stripes, "_curtain_frac": frac, "_curtain_mask": curtain_mask,
        "_image": img,
    }


def save_step_tifs(res, stem, out_dir):
    """Save each pipeline step as a raw TIFF holding the actual data values.

    Unlike a rendered figure, these keep their true numeric range so they open
    correctly in ImageJ/Fiji. Float arrays are written as float32; the binary
    curtain mask as uint8 (0/255). One subfolder per image, one TIFF per step."""
    step_dir = os.path.join(out_dir, stem)
    os.makedirs(step_dir, exist_ok=True)
    steps = {
        "01_original":           np.asarray(res["_image"], dtype=np.float32),
        "02_fft_logpower":       np.asarray(res["_log_power"], dtype=np.float32),
        "03_wedge_fft_logpower": np.asarray(res["_wedge_log_power"], dtype=np.float32),
        "04_stripes":            np.asarray(res["_stripes"], dtype=np.float32),
        "05_curtain_fraction":   np.asarray(res["_curtain_frac"], dtype=np.float32),
        "06_curtain_mask":       (np.asarray(res["_curtain_mask"]).astype(np.uint8) * 255),
    }
    for name, arr in steps.items():
        tifffile.imwrite(os.path.join(step_dir, f"{stem}_{name}.tif"), arr)
    return step_dir


tif_files = find_tifs(INPUT_FOLDER)
print(f"Found {len(tif_files)} TIFF file(s) under {INPUT_FOLDER}")
for p in tif_files[:10]:
    print("  ", os.path.basename(p))
if len(tif_files) > 10:
    print(f"   ... and {len(tif_files) - 10} more")


## Step 1b - Threshold picker (run once to choose `CURTAIN_FRAC_THRESH`)

Overlays the curtain mask on every image for several candidate thresholds and prints the
resulting curtain-area %. Pick the column where the red mask sits on the streaks and stays off
clean tissue/background, then set `CURTAIN_FRAC_THRESH` in Step 0 to that value and re-run.

This cell is **diagnostic only** - it does not affect the CSV.


In [ ]:
CANDIDATE_THRESHOLDS = [0.25, 0.35, 0.50]

# LUT used for the curtain-fraction display (must match the imshow call below).
FRAC_CMAP = "magma"
FRAC_VMIN, FRAC_VMAX = 0.0, 0.6

# Save full-resolution TIFFs so we can build our own publication heatmaps.
# One subfolder per candidate threshold; each holds, for every image, the original,
# the curtain-fraction map, the thresholded texture mask, and the red overlay.
SAVE_PICKER_TIFS = True
PICKER_DIR = os.path.join(OUT_DIR, "threshold_picker")


def _overlay_rgb(img, mask, lo, hi, alpha=0.55):
    """Grayscale image (scaled to [lo, hi]) with the curtain mask blended in red."""
    g = np.clip((np.asarray(img, np.float64) - lo) / (hi - lo + 1e-12), 0.0, 1.0)
    rgb = np.repeat(g[..., None], 3, axis=2)
    m = (np.asarray(mask, np.float64) * alpha)[..., None]
    rgb = rgb * (1.0 - m)
    rgb[..., 0] += m[..., 0]        # push toward red where the mask is set
    return (np.clip(rgb, 0.0, 1.0) * 255).astype(np.uint8)


def _frac_to_rgb(frac, cmap=FRAC_CMAP, vmin=FRAC_VMIN, vmax=FRAC_VMAX):
    """Full-resolution RGB of the curtain-fraction map in the same LUT as the plot."""
    norm = np.clip((np.asarray(frac, np.float64) - vmin) / (vmax - vmin + 1e-12), 0.0, 1.0)
    rgba = plt.get_cmap(cmap)(norm)          # H x W x 4 float in [0, 1]
    return (rgba[..., :3] * 255).astype(np.uint8)


def _colorbar_rgb(height_px, cmap=FRAC_CMAP, vmin=FRAC_VMIN, vmax=FRAC_VMAX,
                  label="curtain fraction", dpi=100):
    """Render a vertical LUT colorbar (with tick labels) as an RGB uint8 array."""
    fig_h = max(height_px / dpi, 1.5)
    fig = plt.figure(figsize=(1.7, fig_h), dpi=dpi)
    cax = fig.add_axes([0.08, 0.06, 0.32, 0.88])
    sm = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax),
                               cmap=plt.get_cmap(cmap))
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label(label, fontsize=11)
    cb.ax.tick_params(labelsize=9)
    fig.canvas.draw()
    rgb = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return rgb


def _pad_to_height(arr, h, fill=255):
    """Vertically center-pad an RGB array to height `h` with `fill`."""
    if arr.shape[0] >= h:
        return arr
    top = (h - arr.shape[0]) // 2
    bot = h - arr.shape[0] - top
    return np.pad(arr, ((top, bot), (0, 0), (0, 0)), constant_values=fill)


def _attach_scale(heatmap_rgb, gap_px=12, **cbar_kw):
    """Append a LUT colorbar to the right of a full-resolution heatmap (RGB)."""
    bar = _colorbar_rgb(heatmap_rgb.shape[0], **cbar_kw)
    h = max(heatmap_rgb.shape[0], bar.shape[0])
    left = _pad_to_height(heatmap_rgb, h)
    right = _pad_to_height(bar, h)
    gap = np.full((h, gap_px, 3), 255, dtype=np.uint8)
    return np.concatenate([left, gap, right], axis=1)


_files = tif_files
if not _files:
    print("No TIFFs found - check INPUT_FOLDER.")
else:
    if SAVE_PICKER_TIFS:
        for t in CANDIDATE_THRESHOLDS:
            os.makedirs(os.path.join(PICKER_DIR, f"threshold_{int(round(t*100))}pct"), exist_ok=True)
        # Standalone LUT reference scale (same magma range as the heatmaps).
        tifffile.imwrite(os.path.join(PICKER_DIR, "curtain_fraction_scale.tif"),
                         _colorbar_rgb(600), photometric="rgb")
        print(f"Saving full-resolution TIFFs under {PICKER_DIR}")
    print(f"{'file':22s}  mean_frac  " + "  ".join(f">{int(t*100)}%" for t in CANDIDATE_THRESHOLDS))
    ncol = 2 + len(CANDIDATE_THRESHOLDS)
    fig, ax = plt.subplots(len(_files), ncol, figsize=(3.0 * ncol, 2.6 * len(_files)))
    if len(_files) == 1:
        ax = ax[None, :]
    for i, path in enumerate(_files):
        stem = os.path.splitext(os.path.basename(path))[0]
        img = load_image(path)
        valid = valid_mask(img); imgf = prep_for_fft(img, valid)
        F, _ = compute_fft(imgf, use_gradient=USE_GRADIENT)
        stripes = recover_stripes(F, wedge_mask(img.shape))
        frac = curtain_fraction_map(stripes, imgf)
        vscore = score_valid(valid)
        lo, hi = np.percentile(img[valid], [1, 99]) if valid.any() else (img.min(), img.max())

        ax[i, 0].imshow(img, cmap="gray", vmin=lo, vmax=hi)
        ax[i, 0].set_ylabel(stem, fontsize=8); ax[i, 0].set_xticks([]); ax[i, 0].set_yticks([])
        if i == 0: ax[i, 0].set_title("original", fontsize=10)
        ax[i, 1].imshow(frac, cmap=FRAC_CMAP, vmin=FRAC_VMIN, vmax=FRAC_VMAX); ax[i, 1].axis("off")
        if i == 0: ax[i, 1].set_title("curtain fraction", fontsize=10)

        pcts = []
        for j, t in enumerate(CANDIDATE_THRESHOLDS):
            a = ax[i, 2 + j]
            mask = (frac > t) & vscore
            pct = 100.0 * mask.sum() / vscore.sum() if vscore.sum() else 0.0
            pcts.append(pct)
            a.imshow(img, cmap="gray", vmin=lo, vmax=hi)
            rgba = np.zeros((*mask.shape, 4)); rgba[..., 0] = 1.0; rgba[..., 3] = mask * 0.55
            a.imshow(rgba); a.axis("off")
            a.set_title((f">{int(t*100)}% texture\n" if i == 0 else "") + f"{pct:.1f}%", fontsize=9)

            if SAVE_PICKER_TIFS:
                tdir = os.path.join(PICKER_DIR, f"threshold_{int(round(t*100))}pct")
                tifffile.imwrite(os.path.join(tdir, f"{stem}_original.tif"),
                                 np.asarray(img, np.float32))
                # Raw curtain-fraction values (float, full precision) for reprocessing.
                tifffile.imwrite(os.path.join(tdir, f"{stem}_curtain_fraction.tif"),
                                 np.asarray(frac, np.float32))
                # Full-resolution heatmap in the same LUT shown in the plot (RGB, publication-ready).
                tifffile.imwrite(os.path.join(tdir, f"{stem}_curtain_fraction_heatmap.tif"),
                                 _frac_to_rgb(frac), photometric="rgb")
                # Same heatmap, but force totally-black (background) pixels to the lowest LUT value.
                frac_bg = np.where(img <= 0, FRAC_VMIN, frac)
                heatmap_bg = _frac_to_rgb(frac_bg)
                tifffile.imwrite(os.path.join(tdir, f"{stem}_curtain_fraction_heatmap_bg_zeroed.tif"),
                                 heatmap_bg, photometric="rgb")
                # Same bg-zeroed heatmap with the LUT reference scale attached on the right.
                tifffile.imwrite(os.path.join(tdir, f"{stem}_curtain_fraction_heatmap_bg_zeroed_with_scale.tif"),
                                 _attach_scale(heatmap_bg), photometric="rgb")
                tifffile.imwrite(os.path.join(tdir, f"{stem}_texture_mask.tif"),
                                 (np.asarray(mask).astype(np.uint8) * 255))
                tifffile.imwrite(os.path.join(tdir, f"{stem}_overlay.tif"),
                                 _overlay_rgb(img, mask, lo, hi))
        mean_frac = frac[vscore].mean() if vscore.any() else np.nan
        print(f"{os.path.basename(path):22s}  {mean_frac:8.3f}   " +
              "  ".join(f"{p:5.1f}" for p in pcts))
    fig.suptitle("Curtain-fraction threshold picker", fontsize=13)
    fig.tight_layout()
    plt.show()


## Step 2 - Batch loop and CSV

Scores every TIFF with the current `CURTAIN_FRAC_THRESH`, saves the optional per-step TIFFs,
and writes one row per image to the CSV in the output folder.


In [ ]:
rows = []
n = len(tif_files)
for k, path in enumerate(tif_files, 1):
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        res = analyze_image(load_image(path))
        row = {
            "file": os.path.basename(path), "path": path,
            "height": res["height"], "width": res["width"], "aspect_ratio": res["aspect_ratio"],
            "curtaining_std": res["curtaining_std"],
            "wedge_energy_ratio": res["wedge_energy_ratio"],
            "curtain_area_pct": res["curtain_area_pct"],
            "mean_curtain_fraction": res["mean_curtain_fraction"],
            "valid_frac": res["valid_frac"],
            "wedge_angle_deg": WEDGE_ANGLE_DEG, "fov_correct": FOV_CORRECT,
            "curtain_frac_thresh": CURTAIN_FRAC_THRESH,
        }
        if SAVE_PICTURES:
            row["step_tifs_dir"] = save_step_tifs(res, stem, PIC_DIR)
        rows.append(row)
        print(f"[{k}/{n}] {os.path.basename(path):22s} "
              f"wedge_ratio={res['wedge_energy_ratio']:6.2f}  "
              f"curtain_area={res['curtain_area_pct']:5.1f}%  "
              f"mean_frac={res['mean_curtain_fraction']:.3f}")
    except Exception as exc:
        print(f"[{k}/{n}] {os.path.basename(path):22s} FAILED: {exc}")
        rows.append({"file": os.path.basename(path), "path": path,
                     "wedge_energy_ratio": np.nan, "curtain_area_pct": np.nan,
                     "wedge_angle_deg": WEDGE_ANGLE_DEG, "fov_correct": FOV_CORRECT,
                     "curtain_frac_thresh": CURTAIN_FRAC_THRESH})

df = pd.DataFrame(rows)
out_csv = os.path.join(OUT_DIR, CSV_NAME)
df.to_csv(out_csv, index=False)
print(f"\nSaved {len(df)} row(s) to: {out_csv}")
if SAVE_PICTURES:
    print(f"Step TIFFs saved under: {PIC_DIR}")
df
